In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum, count, avg, round as _round, row_number
from pyspark.sql.window import Window

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas7_PySpark_ETL") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/09 13:50:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/09 13:50:08 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
# A. EXTRACT
# Membaca data dari HDFS
hdfs_base_path = "hdfs://localhost:9000/user/mahasiswa/tugas7/raw"

df_transaksi = spark.read.csv(f"{hdfs_base_path}/tugas7_transaksi.csv", header=True, inferSchema=True)
df_cabang = spark.read.json(f"{hdfs_base_path}/tugas7_cabang.json")
df_menu = spark.read.csv(f"{hdfs_base_path}/tugas7_menu.csv", header=True, inferSchema=True)

# Menampilkan schema dan jumlah baris untuk memverifikasi
print("Data Transaksi Count:", df_transaksi.count())
print("Data Cabang Count   :", df_cabang.count())
print("Data Menu Count     :", df_menu.count())

Data Transaksi Count: 4000
Data Cabang Count   : 3
Data Menu Count     : 15


In [3]:
# B. TRANSFORM - Join & Kolom Turunan
# Menggabungkan ketiga dataframe
df_gabungan = df_transaksi.join(df_cabang, on="cabang_id", how="inner") \
                          .join(df_menu, on="menu_id", how="inner")

# Menambahkan kolom total_penjualan (qty * harga)
df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))

# Tampilkan 5 baris pertama
df_gabungan.show(5)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


In [4]:
# C. TRANSFORM - Window Function
# 1. Agregasi total penjualan per cabang dan menu
df_menu_cabang = df_gabungan.groupBy("nama_cabang", "nama_menu") \
    .agg(_sum("total_penjualan").alias("total_penjualan"))

# 2. Definisikan Window Specification (dipartisi per cabang, diurutkan total_penjualan descending)
window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())

# 3. Terpakan row_number() dan filter top-2
df_top2_menu = df_menu_cabang \
    .withColumn("rn", row_number().over(window_cabang)) \
    .filter(col("rn") <= 2) \
    .drop("rn")

df_top2_menu.show()

[Stage 21:>                                                         (0 + 1) / 1]

+--------------------+---------+---------------+
|         nama_cabang|nama_menu|total_penjualan|
+--------------------+---------+---------------+
|Kopi Nusantara Ma...|   Menu-1|        8505000|
|Kopi Nusantara Ma...|  Menu-12|        8085000|
|Kopi Nusantara Se...|  Menu-13|        8750000|
|Kopi Nusantara Se...|   Menu-9|        8715000|
|Kopi Nusantara Yo...|  Menu-12|        9275000|
|Kopi Nusantara Yo...|  Menu-15|        8160000|
+--------------------+---------+---------------+



In [5]:
# D. TRANSFORM - SparkSQL
# Buat Temporary View
df_gabungan.createOrReplaceTempView("v_tugas7")

# Query Spark SQL murni
query_barista = """
SELECT 
    kepala_barista, 
    SUM(total_penjualan) AS total_penjualan
FROM v_tugas7
GROUP BY kepala_barista
ORDER BY total_penjualan DESC
"""

df_barista_sales = spark.sql(query_barista)
df_barista_sales.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



In [6]:
# E. LOAD
# Path penyimpanan HDFS
path_output_parquet = "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"

# Simpan ke HDFS dipartisi berdasarkan kategori_menu
df_gabungan.write \
    .mode("overwrite") \
    .partitionBy("kategori_menu") \
    .parquet(path_output_parquet)

print("Data berhasil disimpan ke HDFS dalam format Parquet!")

# Verifikasi dengan membaca kembali data parquet dan menghitung jumlah baris
df_verifikasi = spark.read.parquet(path_output_parquet)
print("Jumlah baris data verifikasi Parquet:", df_verifikasi.count())

Data berhasil disimpan ke HDFS dalam format Parquet!
Jumlah baris data verifikasi Parquet: 4000


In [7]:
# F. Ringkasan Eksekutif & Rekomendasi
df_ringkasan = df_gabungan.groupBy("nama_cabang").agg(
    _sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    _round(avg("total_penjualan"), 2).alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan").desc())

df_ringkasan.show()

+--------------------+---------------+----------------+-------------------------+
|         nama_cabang|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+--------------------+---------------+----------------+-------------------------+
|Kopi Nusantara Se...|       90542000|            1393|                 64997.85|
|Kopi Nusantara Ma...|       87138000|            1302|                 66926.27|
|Kopi Nusantara Yo...|       84645000|            1305|                 64862.07|
+--------------------+---------------+----------------+-------------------------+



1. **Kinerja Cabang Terbaik**:  
   Berdasarkan agregasi total nilai penjualan dan volume transaksi, cabang dengan kinerja terbaik secara keseluruhan adalah Kopi Nusantara Semarang (dikepalai oleh Kepala Barista Wulan). Cabang ini berhasil mencatatkan nilai pendapatan total dan rata-rata nilai transaksi (average transaction value) tertinggi dibandingkan cabang lainnya. Hal ini didorong oleh tingginya volume permintaan pada produk kategori Kopi dan Makanan Ringan.

2. **Rekomendasi Strategis untuk Cabang Kinerjanya Terlemah**:  
   Cabang dengan kinerja terlemah adalah Kopi Nusantara Magelang. Untuk meningkatkan performa penjualan di cabang ini, disarankan untuk melakukan promosi lebih gencar pada menu-menu populer berdasarkan hasil analisis *Window Function* (Bagian C), yaitu Menu-14 dan Menu-8 (kategori *Non-Kopi* dan *Makanan Ringan*). 

3. **Alasan Strategis**:  
   Penjualan produk non-kopi dan camilan pendamping menunjukkan tren margin positif dan peminat yang stabil di cabang Magelang. Mengombinasikan menu terlaris tersebut ke dalam paket *bundling* promosi hemat (misalnya *Bundle Kopi + Makanan Ringan*) akan meningkatkan nilai rata-rata transaksi per pelanggan (*basket size*), sekaligus menarik minat segmen pelanggan baru yang tidak mengonsumsi kafein.

In [8]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
